In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

df = pd.read_csv('data/EmendasParlamentares.csv', sep = ';', encoding='latin-1')
print(f"O arquivo possui {len(df)} linhas e {len(df.columns)} colunas.")
df.head()

O arquivo possui 89737 linhas e 28 colunas.


C:\Users\hboni\AppData\Local\Temp\ipykernel_11080\1509812720.py:5: DtypeWarning: Columns (0: Código da Emenda, 1: Código do Autor da Emenda, 2: Número da emenda) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/EmendasParlamentares.csv', sep = ';', encoding='latin-1')


,Código da Emenda,Ano da Emenda,Tipo de Emenda,Código do Autor da Emenda,Nome do Autor da Emenda,Número da emenda,Localidade de aplicação do recurso,Código Município IBGE,Município,Código UF IBGE,...,Código Ação,Nome Ação,Código Plano Orçamentário,Nome Plano Orçamentário,Valor Empenhado,Valor Liquidado,Valor Pago,Valor Restos A Pagar Inscritos,Valor Restos A Pagar Cancelados,Valor Restos A Pagar Pagos
0,Sem informação,2014,Emenda Individual - Transferências com Finalid...,S/I,Sem informação,S/I,MUNDO NOVO - MS,5005681,MUNDO NOVO,5000000,...,5450,IMPLANTACAO E MODERNIZACAO DE INFRAESTRUTURA P...,0000,IMPLANTACAO E MODERNIZACAO DE INFRAESTRUTURA P...,"645000,00","0,00","0,00","0,00","0,00","645000,00"
1,Sem informação,2014,Emenda Individual - Transferências com Finalid...,S/I,Sem informação,S/I,SÃO GONÇALO - RJ,3304904,SÃO GONÇALO,3300000,...,20RP,APOIO A INFRAESTRUTURA PARA A EDUCACAO BASICA,0000,INFRAESTRUTURA PARA A EDUCACAO BASICA - DESPES...,"179919,36","0,00","0,00","0,00","179919,36","0,00"
2,Sem informação,2014,Emenda Individual - Transferências com Finalid...,S/I,Sem informação,S/I,CASTRO - PR,4104907,CASTRO,4100000,...,8581,ESTRUTURACAO DA REDE DE SERVICOS DE ATENCAO PR...,0000,ESTRUTURACAO DA REDE DE SERVICOS DE ATENCAO BA...,"858000,00","0,00","0,00","0,00","0,00","858000,00"
3,Sem informação,2014,Emenda Individual - Transferências com Finalid...,S/I,Sem informação,S/I,GOIÁS (UF),Sem informação,Sem informação,5200000,...,8535,ESTRUTURACAO DE UNIDADES DE ATENCAO ESPECIALIZ...,0000,ESTRUTURACAO DE UNIDADES DE ATENCAO ESPECIALIZ...,"8281302,28","0,00","0,00","0,00","400000,00","7881302,28"
4,Sem informação,2014,Emenda Individual - Transferências com Finalid...,S/I,Sem informação,S/I,BOM JESUS - RN,2401701,BOM JESUS,2400000,...,5450,IMPLANTACAO E MODERNIZACAO DE INFRAESTRUTURA P...,0000,IMPLANTACAO E MODERNIZACAO DE INFRAESTRUTURA P...,"292500,00","0,00","0,00","0,00","0,00","292500,00"


In [2]:
df['Valor Empenhado'] = df['Valor Empenhado'].astype(str).str.replace(',','', regex=False).str.replace('.','.', regex=False)
df['Valor Empenhado'] = pd.to_numeric(df['Valor Empenhado'], errors='coerce').fillna(0)

df['Nome do Autor da Emenda'] = df['Nome do Autor da Emenda'].str.upper().str.strip()
df['Nome Função'] = df['Nome Função'].str.upper().str.strip()
df = df.replace('Sem Informação', np.nan)

colunas_selecionadas = ['Ano da Emenda', 'Tipo de Emenda', 'Nome do Autor da Emenda', 'Município', 'UF', 'Nome Função', 'Valor Empenhado']
df_final = df[colunas_selecionadas].copy()

def definir_tipo_autor(nome):
    nome = str(nome).upper()
    if 'BANCADA' in nome: return 'Bancada'
    if 'COM.' in nome or 'COMISSÃO' in nome: return 'Comissão'
    if 'RELATOR' in nome: return 'Relator'
    if 'SEM INFORMAÇÃO' in nome: return 'Sem Informação'
    return 'Individual'

df_final['Tipo de Autor'] = df_final['Nome do Autor da Emenda'].apply(definir_tipo_autor)

df_final['Ano da Emenda'] = pd.to_numeric(df_final['Ano da Emenda'], errors='coerce')
df_final = df_final.dropna(subset=['Ano da Emenda'])
df_final = df_final[(df_final['Ano da Emenda'] >= 2019) & (df_final['Ano da Emenda'] <= 2025)]

print("Limpeza e Categorização Concluídas com Sucesso!")

Limpeza e Categorização Concluídas com Sucesso!


In [3]:
engine = create_engine('sqlite:///transparencia_emendas.db')
df_final.to_sql('emendas_parlamentares', con=engine, if_exists='replace', index=False)

print("Dados inseridos no banco de dados com sucesso!")

Dados inseridos no banco de dados com sucesso!


In [4]:
termos_para_remover_uf = ['SEM INFORMAÇÃO', 'Múltiplo']
df_uf = df_final[~df_final['UF'].str.contains('|'.join(termos_para_remover_uf), case=False, na=False)]

verba_por_uf = df_uf.groupby('UF')['Valor Empenhado'].sum().sort_values(ascending=False)
print("\nVerba por UF (Top 10):\n")
for nome, valor in verba_por_uf.head(10).items():
    print(f"{nome}: R$ {valor:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))


Verba por UF (Top 10):

SÃO PAULO: R$ 1.465.002.667.828,00
MINAS GERAIS: R$ 1.137.308.292.258,00
RIO DE JANEIRO: R$ 955.128.518.699,00
BAHIA: R$ 937.053.064.017,00
RIO GRANDE DO SUL: R$ 780.882.336.588,00
PARANÁ: R$ 701.898.984.116,00
PERNAMBUCO: R$ 679.107.874.723,00
CEARÁ: R$ 652.521.198.753,00
MARANHÃO: R$ 584.442.425.236,00
GOIÁS: R$ 566.223.235.819,00


In [ ]:
df_final.to_csv('data/EmendasParlamentares_Limpo.csv', index=False, encoding='UTF-8')
print("Arquivo CSV limpo salvo com sucesso!")